In [0]:
CREATE OR REPLACE TABLE workspace.dataforge.dfa_insurance_project_raw_dataset AS
SELECT * FROM read_files(
  '/Volumes/workspace/dataforge/final_project/dfa_insurance_project_raw_dataset.csv',
  format => 'csv',
  header => true,
  inferSchema => true
)
-- SELECT * FROM dfa_insurance_project_raw_dataset

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.dataforge.dfa_insurance_project_raw_dataset

Customer_ID,Age,Gender,Province,Monthly_Income,Join_Date,Policy_ID,Policy_Type,Premium_Amount,Policy_Status,Claim_ID,Claim_Date,Claim_Amount,Claim_Status,Fraud_Flag,_rescued_data
CUST1000,44,Male,Gauteng,26064,2020-05-30,POL5000,Home,1495,Active,CLM9000,2024-04-14,20438.99,Approved,No,null
CUST1001,41,Male,Free State,3500.0,2025-06-16,POL5001,Auto,746.28,Active,CLM9001,2025-09-08,4583.11,approved,No,null
CUST1002,33,Female,Limpopo,29552.92,2020-06-14,POL5002,Health,2325.42,Active,CLM9002,12-Jun-2024,11892.02,Rejected,No,null
CUST1003,41,Female,western cape,3500.0,2024-06-10,POL5003,Auto,885.13,Active,CLM9003,2025-01-24,15139.51,Approved,No,null
CUST1004,42,Female,Mpumalanga,3500.0,2024-07-15,POL5004,Home,"R 1,875.25",Active,CLM9004,2025/01/22,29975,Approved,0,null
CUST1005,33,Female,Limpopo,null,2024-04-13,POL5005,Home,2210.31,Active,CLM9005,2024-10-23,30336.75,Approved,No,null
CUST1006,38,Male,Free State,49905.01,2022-12-19,POL5006,Motor,433.09,Active,CLM9006,2025/01/14,500.0,Approved,No,null
CUST1007,30,FEMALE,KwaZulu-Natal,20975.4,10-14-2021,POL5007,Life,817.56,Active,CLM9007,2024-06-16,39102.08,Rejected,No,null
CUST1008,42,Male,North West,31654.77,2022-03-28,POL5008,Home,"R 1,410.71",Active,CLM9008,2024-02-02,48634.23,Approved,No,null
CUST1009,41,Male,Limpopo,26957.76,2020-11-02,POL5009,Funeral,326.92,Active,CLM9009,2024-08-05,null,Approved,No,null


In [0]:
SELECT Count(DISTINCT(Customer_ID)) AS Total_number_of_customers FROM workspace.dataforge.dfa_insurance_project_raw_dataset


Total_number_of_customers
518


In [0]:
SELECT COUNT(DISTINCT(TRIM(Policy_ID))) AS Total_number_of_Policies FROM workspace.dataforge.dfa_insurance_project_raw_dataset


Total_number_of_Policies
520


In [0]:
SELECT COUNT(Claim_ID) AS Total_number_of_Claims FROM workspace.dataforge.dfa_insurance_project_raw_dataset

Total_number_of_Claims
547


In [0]:
SELECT
SUM(CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE)) AS Total_Claim_Amount FROM workspace.dataforge.dfa_insurance_project_raw_dataset

Total_Claim_Amount
1.2578657020740738E7


In [0]:
SELECT DISTINCT(TRIM(UPPER(Policy_Type))) As Policy_Type, count(TRIM(UPPER(Claim_ID))) AS Total_number_of_Policies_Claims FROM workspace.dataforge.dfa_insurance_project_raw_dataset GROUP BY TRIM(UPPER(Policy_Type))

Policy_Type,Total_number_of_Policies_Claims
HOME,94
AUTO,128
HEALTH,134
MOTOR,10
LIFE,105
FUNERAL,68
MEDICAL,5
PROPERTY,3


In [0]:
SELECT TRIM(UPPER(Policy_Type)) AS Policy_Type,
MEAN(CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE)) AS Average_claim_amount_per_policy_type 

 FROM workspace.dataforge.dfa_insurance_project_raw_dataset GROUP BY TRIM(UPPER(Policy_Type))

Policy_Type,Average_claim_amount_per_policy_type
FUNERAL,14198.578531590416
HOME,25467.256000779722
MEDICAL,17714.302
MOTOR,14929.177
HEALTH,12234.462611940296
LIFE,45518.0961925926
AUTO,19049.831233048422
PROPERTY,20201.643333333333


In [0]:
SELECT DISTINCT TRIM(coalesce(CASE WHEN TRIM(Fraud_Flag) like 0 THEN 'No'
WHEN TRIM(Fraud_Flag) like 1 THEN "Yes"
WHEN TRIM(Fraud_Flag) like "Y%%" THEN "Yes"
WHEN TRIM(Fraud_Flag) like "N%" THEN "No"
WHEN TRIM(Fraud_Flag) like "N" THEN "No"
WHEN TRIM(Fraud_Flag) like "no" THEN "No"
ELSE TRIM(Fraud_Flag)
END,
(SELECT MODE(TRIM(Fraud_Flag)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Fraud_Flag, 
COUNT(TRIM(coalesce(CASE WHEN TRIM(Fraud_Flag) like 0 THEN 'No'
WHEN TRIM(Fraud_Flag) like 1 THEN "Yes"
WHEN TRIM(Fraud_Flag) like "Y%%" THEN "Yes"
WHEN TRIM(Fraud_Flag) like "N%" THEN "No"
WHEN TRIM(Fraud_Flag) like "N" THEN "No"
WHEN TRIM(Fraud_Flag) like "no" THEN "No"
ELSE TRIM(Fraud_Flag)
END,
(SELECT MODE(TRIM(Fraud_Flag)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset)))) AS Number_of_fraud_vs_non_fraud_claims
FROM workspace.dataforge.dfa_insurance_project_raw_dataset
GROUP BY TRIM(coalesce(CASE WHEN TRIM(Fraud_Flag) like 0 THEN 'No'
WHEN TRIM(Fraud_Flag) like 1 THEN "Yes"
WHEN TRIM(Fraud_Flag) like "Y%%" THEN "Yes"
WHEN TRIM(Fraud_Flag) like "N%" THEN "No"
WHEN TRIM(Fraud_Flag) like "N" THEN "No"
WHEN TRIM(Fraud_Flag) like "no" THEN "No"
ELSE TRIM(Fraud_Flag)
END,
(SELECT MODE(TRIM(Fraud_Flag)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset)))

Fraud_Flag,Number_of_fraud_vs_non_fraud_claims
Yes,79
No,471


In [0]:
SELECT DISTINCT TRIM(Claim_ID) AS Claim_ID,
SUM(CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE)) AS Top_5_highest_claim_amounts   

 FROM workspace.dataforge.dfa_insurance_project_raw_dataset GROUP BY TRIM(Claim_ID)
 ORDER BY TRIM(Claim_ID) DESC
 LIMIT 5

Claim_ID,Top_5_highest_claim_amounts
CLM9519,45945.0
CLM9518,16166.05
CLM9517,21517.39
CLM9516,7581.91
CLM9515,38996.0


In [0]:
CREATE OR REPLACE TEMPORARY VIEW Reference AS
SELECT TRIM(coalesce(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END,
(SELECT mode(trim(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Province
FROM workspace.dataforge.dfa_insurance_project_raw_dataset;

SELECT DISTINCT Province, COUNT(Province) AS Claims_per_location
FROM Reference GROUP BY Province 

Province,Claims_per_location
Gauteng,64
Limpopo,47
Eastern Cape,75
North West,72
Free State,72
Mpumalanga,55
Western Cape,75
KwaZulu-Natal,90


In [0]:
SELECT TRIM(coalesce(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END,
(SELECT mode(trim(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS Province,

COUNT(TRIM(coalesce(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END,
(SELECT mode(trim(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset)))) AS Claims_per_location
FROM workspace.dataforge.dfa_insurance_project_raw_dataset 

GROUP BY TRIM(coalesce(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END,
(SELECT mode(trim(CASE
WHEN Trim(UPPER(Province)) IN ("GP", "GAUTENG") THEN "Gauteng"
WHEN Trim(UPPER(Province)) IN ("EC", "EASTERN CAPE") THEN "Eastern Cape"
WHEN Trim(UPPER(Province)) IN ("FS", "FREE STATE") THEN "Free State"
WHEN Trim(UPPER(Province)) IN ("NW", "NORTH WEST") THEN "North West"
WHEN Trim(UPPER(Province)) IN ("WC", "WESTERN CAPE") THEN "Western Cape"
WHEN Trim(UPPER(Province)) IN ("MP", "MPUMALANGA") THEN "Mpumalanga"
WHEN Trim(UPPER(Province)) IN ("LP", "LIMPOPO") THEN "Limpopo"
WHEN Trim(UPPER(Province)) IN ("KZN", "KWAZULU-NATAL", "KWAZULU NATAL") THEN "KwaZulu-Natal"
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset)))

Province,Claims_per_location
Gauteng,64
Limpopo,47
Eastern Cape,75
North West,72
Free State,72
Mpumalanga,55
Western Cape,75
KwaZulu-Natal,90


In [0]:
SELECT DISTINCT Customer_ID,
SUM(CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE) END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE)) AS Claim_Amount
FROM workspace.dataforge.dfa_insurance_project_raw_dataset
GROUP BY Customer_ID
Order by CAST(TRIM(coalesce(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE) END,
(SELECT mean(Trim(CASE 
  WHEN Claim_Amount LIKE '%R%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '[R, ]', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%,%' THEN CAST(REGEXP_REPLACE(Claim_Amount, ',', '') AS DOUBLE)
  WHEN Claim_Amount LIKE '%-%' THEN CAST(REGEXP_REPLACE(Claim_Amount, '-', '') AS DOUBLE)
  ELSE CAST(Claim_Amount AS DOUBLE)
END)) FROM workspace.dataforge.dfa_insurance_project_raw_dataset))) AS DOUBLE) DESC
LIMIT 5

Customer_ID,Claim_Amount
CUST1050,143857.6
CUST1324,115579.76
CUST1429,97527.12
CUST1425,92918.98
CUST1068,89909.74


In [0]:
SELECT
CASE
WHEN substring(trim(Join_Date),3,1) = "/" THEN to_date(trim(Join_Date), 'dd/MM/yyyy')
WHEN substring(trim(Join_Date),5,1) = "-" THEN to_date(trim(Join_Date), 'yyyy-MM-dd')
WHEN substring(trim(Join_Date),5,1) = "/" THEN to_date(trim(Join_Date), 'yyyy/MM/dd')
WHEN trim(Join_Date) LIKE '__-___-____' THEN to_date(trim(Join_Date), 'dd-MMM-yyyy')
WHEN substring(trim(Join_Date),3,1) = "-" THEN to_date(trim(Join_Date), 'MM-dd-yyyy')
END AS Join_Date,
COUNT(CASE
WHEN substring(trim(Join_Date),3,1) = "/" THEN to_date(trim(Join_Date), 'dd/MM/yyyy')
WHEN substring(trim(Join_Date),5,1) = "-" THEN to_date(trim(Join_Date), 'yyyy-MM-dd')
WHEN substring(trim(Join_Date),5,1) = "/" THEN to_date(trim(Join_Date), 'yyyy/MM/dd')
WHEN trim(Join_Date) LIKE '__-___-____' THEN to_date(trim(Join_Date), 'dd-MMM-yyyy')
WHEN substring(trim(Join_Date),3,1) = "-" THEN to_date(trim(Join_Date), 'MM-dd-yyyy')
END) AS  Claims_trend_over_time
FROM workspace.dataforge.dfa_insurance_project_raw_dataset
GROUP BY CASE
WHEN substring(trim(Join_Date),3,1) = "/" THEN to_date(trim(Join_Date), 'dd/MM/yyyy')
WHEN substring(trim(Join_Date),5,1) = "-" THEN to_date(trim(Join_Date), 'yyyy-MM-dd')
WHEN substring(trim(Join_Date),5,1) = "/" THEN to_date(trim(Join_Date), 'yyyy/MM/dd')
WHEN trim(Join_Date) LIKE '__-___-____' THEN to_date(trim(Join_Date), 'dd-MMM-yyyy')
WHEN substring(trim(Join_Date),3,1) = "-" THEN to_date(trim(Join_Date), 'MM-dd-yyyy')
END
ORDER BY Join_Date ASC

Join_Date,Claims_trend_over_time
2020-02-29,1
2020-05-05,1
2020-05-18,1
2020-05-27,1
2020-05-30,1
2020-06-04,1
2020-06-14,1
2020-06-30,2
2020-07-08,1
2020-07-15,1
